# Customer‑Support Chatbot for an E-Commerce Store

Build a **chatbot** that answers customer service questions about Everstorm Outfitters, an imaginary e-commerce store.



## Architecture
* **Ingest & chunk** unstructured docs  
* **Embed** chunks and **index** with *FAISS*  
* **Retrieve** context and **craft prompts**  
* **Run** an open‑weight LLM locally with *Ollama*  
* **Build** a Retrieval-Augmented Generation (RAG) chain
* **Package** the chat loop in a minimal **Streamlit** web UI

## Steps


1. **Environment setup**
2. **Data preparation**  
   a. Load source documents  
   b. Chunk the text  
3. **Build a retriever**  
   a. Generate embeddings  
   b. Build the FAISS vector index  
4. **Build a generation engine**. Load the *Gemma3-1B* model through Ollama and run a sanity check.  
5. **Build a RAG**. Connect the system prompt, retriever, and LLM together. 
6. **Streamlit UI**. Wrap everything in a simple web app so users can chat with the bot.


## 1 - Environment setup

We use conda to manage our project dependencies and ensure everyone has a consistent setup. Conda is an open-source package and environment manager that makes it easy to install libraries and switch between isolated environments. https://docs.conda.io/en/latest/

Create and activate a clean *conda* environment and install the required packages. If you don't have conda installed, visit https://www.anaconda.com/docs/getting-started/miniconda/main.


Open your terminal, navigate to the project folder where this notebook is located, and run the following commands.

```bash
conda env create -f environment.yml && conda activate rag-chatbot

# (Optional but recommended) Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=rag-chatbot --display-name "rag-chatbot"
```
Once this is done, you can select “rag-chatbot” from the Kernel → Change Kernel menu in Jupyter or VS Code.




Import required libraries

In [1]:
# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector store to store and retrieve embeddings efficiently using FAISS
from langchain.vectorstores import FAISS

# Generate text embeddings using OpenAI or Hugging Face models
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings, SentenceTransformerEmbeddings

# Use local LLMs (e.g., via Ollama) for response generation
from langchain.llms import Ollama

# Build a retrieval chain that combines a retriever, a prompt, and an LLM
from langchain.chains import ConversationalRetrievalChain

# Create prompts for the RAG system
from langchain.prompts import PromptTemplate

print("✅ Libraries imported - you're good to go!")

/Users/julieszhu/miniconda3/envs/rag-chatbot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported - you're good to go!


## 2 - Data preparation
This step is to turn all reference documents into small chunks of text that a retriever can index and search. These documents consist of:
* PDF files: local documents such as policies, user manuals, or guides.
* Web pages (HTML): online documentation, blog posts, or help articles.

Perform two actions:
* **Ingesting**: load every PDF and collect the raw text in a list named `raw_docs`.
* **Chunking**: split each document into small, overlapping chunks so later steps can match a user query to the most relevant passage.

### 2.1 - Ingest source documents

Use PyPDFLoader from LangChain, which makes it easy to extract text from PDF files for downstream processing. To learn more about how to use it, refer to: https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/

Use **PyPDFLoader** to load every PDF whose filename matches `data/Everstorm_*.pdf` and collect all pages in a list called `raw_docs`. The content of these PDFs is synthetically and downloaded online.

In [19]:
pdf_paths = glob.glob("data/Everstorm_*.pdf")

raw_docs = []
for path in pdf_paths:
    raw_docs.extend(PyPDFLoader(path).load())

print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")

Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong pointing object 76 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)


Loaded 8 PDF pages from 4 files.


### 2.2 - Chunk the text

Chuck long documents can improve greatly LLM processing efficiency.

We split large documents into smaller, overlapping chunks. We use this library `RecursiveCharacterTextSplitter` from LangChain, which splits text intelligently while keeping paragraph or sentence boundaries intact.  visit: https://python.langchain.com/api_reference/text_splitters/character/langchain_text_splitters.character.RecursiveCharacterTextSplitter.html

We split each document into chunks of roughly 300 tokens with a 30-token overlap using `RecursiveCharacterTextSplitter`. This overlap helps maintain continuity across chunks while ensuring each piece stays small enough for embedding and retrieval.

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(raw_docs)
print(f"✅ {len(chunks)} chunks ready for embedding")

✅ 42 chunks ready for embedding


## 3 -Build embedding and vector DB


1. **Load a model to generate embeddings**: convert each text chunk from the reference documents into a fixed‑length vector that captures its semantic meaning.  
2. **Build vector database**: store these embeddings in a vector database.


### 3.1 - Load a model to generate embeddings

Convert each doc into embeddings.


We use the smaller gte-small model for simplicity and reproducibility. Use imported `SentenceTransformerEmbeddings` library to load the model and use it to embed queries. To learn more about lagnchain's embedding support, visit: https://python.langchain.com/docs/integrations/text_embedding/

In [21]:
emb_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-small")

# Single embedding for one sentence
embedding_vector = emb_model.embed_query("Hello world!")
print(len(embedding_vector))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6012.28it/s]
BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


### 3.2 - Build a vector database

After embeddings are generated via embedding model, we store embeddings in a vector db.

We use open source vector db **FAISS**. FAISS is optimized for fast nearest-neighbor search in high-dimensional spaces. https://github.com/facebookresearch/faiss/wiki/getting-started

We ingest all document embeddings into FAISS, which builds an in-memory vector index. This index allows us to efficiently query for the *k* most similar chunks to any given question.

During inference, we’ll use this index to retrieve the top-k relevant chunks and pass them to the LLM as context, enabling it to answer questions grounded in our documents.



In [22]:
vectordb = FAISS.from_documents(chunks, emb_model)
vectordb.save_local("faiss_index")
print("✅ Vector store with", vectordb.index.ntotal, "embeddings")

✅ Vector store with 42 embeddings


## 4 - LLM processing
At the core of any RAG system lies an **LLM**. After locating relevant information, LLM use that information to generate semnatic responses.

We use **Gemma 3* (1B), a small but capable open-weight model, and run it entirely on local machine using Ollama. 

To learn more about Ollama, visit: https://github.com/ollama/ollama. 


### 4.1 - Install `ollama` and serve `gemma3`

Follow these steps to set up Ollama and start the model server:

**1 - Install**
```bash
# macOS (Homebrew)
brew install ollama
# Linux
curl -fsSL https://ollama.com/install.sh | sh
```

If you’re on Windows, install using the official installer from https://ollama.com/download.

**2 - Start the Ollama server (keep this terminal open)**
```bash
ollama serve
```
This command launches a local server at http://localhost:11434, which will stay running in the background.


**3 - Pull the Gemma mode (or the model of your choice) in a new terminal**
```bash
ollama pull gemma3:1b
```

This downloads the 1B version of Gemma 3, a compact model suitable for running on most modern laptops. Once downloaded, Ollama will automatically handle model loading and caching.


After this setup, your system is ready to generate responses locally using the Gemma model through the Ollama API.


### 4.2 - Test LLM with a random prompt


In [10]:
llm = Ollama(model="gemma3:1b", temperature=0.1)
print(llm.invoke("Tell me one fun fact about googler."))

Okay, here's a fun fact about Google that you might not know:

**Google employees are allowed to have a "digital detox" – a period where they completely disconnect from all work-related technology for a few hours each day.** 

Seriously! They've implemented policies and encouraged it to help with mental health and productivity. It's a pretty radical idea for a company that thrives on constant connectivity! 

---

Would you like to know another fun fact?


## Build a RAG

### 5.1 - Define a system prompt

The **system prompt** instruct the model to answer only using the retrieved context and to admit when it doesn’t know the answer to avoid hallucination .


In [11]:
SYSTEM_TEMPLATE = """
You are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.
If the answer is not in CONTEXT, respond with “I'm not sure from the docs.”

Rules:
1) Use ONLY the provided <context> to answer.
2) If the answer is not in the context, say: "I don't know based on the retrieved documents."
3) Be concise and accurate. Prefer quoting key phrases from the context.
4) When possible, cite sources as [source: source] using the metadata.

CONTEXT:
{context}

USER:
{question}
"""

### 5.2 Create a RAG chain
We can connect vector db, prompt and llm into a single RAG pipeline. Vector db provides the most relevant chunks from vector index, the prompt injects those chunks into the system message, and the LLM uses that context to produce the final answer. 

This connection is handled through LangChain’s `ConversationalRetrievalChain`, which combines retrieval and generation.  visit: https://python.langchain.com/api_reference/langchain/chains/langchain.chains.conversational_retrieval.base.ConversationalRetrievalChain.html

In [23]:
from langchain.prompts import ChatPromptTemplate


In [24]:
prompt = ChatPromptTemplate.from_template(SYSTEM_TEMPLATE)
p = prompt.format(context="my context", question="what is your name")
p

'Human: \nYou are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.\nIf the answer is not in CONTEXT, respond with “I\'m not sure from the docs.”\n\nRules:\n1) Use ONLY the provided <context> to answer.\n2) If the answer is not in the context, say: "I don\'t know based on the retrieved documents."\n3) Be concise and accurate. Prefer quoting key phrases from the context.\n4) When possible, cite sources as [source: source] using the metadata.\n\nCONTEXT:\nmy context\n\nUSER:\nwhat is your name\n'

In [25]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(SYSTEM_TEMPLATE)
retriever = vectordb.as_retriever(search_kwargs={"k": 8})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def rag_step(question: str):
    docs = retriever.get_relevant_documents(question)
    context = format_docs(docs)

    messages = prompt.format_messages(context=context, question=question)
    answer = llm.invoke(messages)

    return {"answer": answer, "source_documents": docs}

When you ask a question, the retriever pulls the top few relevant text chunks, the model reads them through the system prompt, and then it generates an answer based on that context.




### 5.3 - Validate the RAG chain

We run a few questions to make sure everything behaves as expecte. Experiment by adding you own questions.

In [26]:
test_questions = [
    "If I'm not happy with my purchase, what is your refund policy and how do I start a return?",
    "How long will delivery take for a standard order, and where can I track my package once it ships?",
    "What's the quickest way to contact your support team, and what are your operating hours?",
]

# chat_history = []
for q in test_questions:
    result = rag_step(q)
    print(f"\nQ: {q}\nA: {result['answer'][:350]}...")


Q: If I'm not happy with my purchase, what is your refund policy and how do I start a return?
A: Okay, here’s your refund policy and return process:

**Our Refund Policy**

We offer refunds for defective or unsatisfactory items within 30 days of delivery. Here’s a breakdown:

*   **Eligible Returns:** Returns are accepted if the item is defective or doesn’t meet your expectations.
*   **Refund Timeline:**
    *   Warehouse receipt → inspection...

Q: How long will delivery take for a standard order, and where can I track my package once it ships?
A: The standard delivery time for a standard order is 7.95 hours, and you can track your package once it ships at [live-rate] and [free]. You can track your package at [tracking link] which is emailed upon label creation....

Q: What's the quickest way to contact your support team, and what are your operating hours?
A: You can contact logistics@everstorm.example within 30 minutes of placing your order. They operate from 8:00 AM to 18:00 PM Mo

### 6 - Build the Streamlit UI 

A lightweight UI as interface only to demonstrate the end-to-end flow.

Streamlit is a common choice for fast prototyping because it lets you make a usable interface with only a few lines of Python.  Streamlit Quickstart: https://docs.streamlit.io/deploy/streamlit-community-cloud/get-started/quickstart


In this cell, we write a minimal **`app.py`** that starts a simple chat UI and calls your RAG chain.

In [28]:
%%bash
cat > app.py <<'PY'
import pathlib, streamlit as st
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import Ollama
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

st.set_page_config(page_title="Customer Support Chatbot")
st.title("Customer Support Chatbot")

@st.cache_resource
def init_chain():
    vectordb = FAISS.load_local(
        "faiss_index",
        HuggingFaceEmbeddings(model_name="thenlper/gte-small"),
        allow_dangerous_deserialization=True,
    )
    retriever = vectordb.as_retriever(search_kwargs={"k": 8})
    llm = Ollama(model="gemma3:1b", temperature=0.1)

    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
    )

    return ConversationalRetrievalChain.from_llm(
        llm,
        retriever,
        memory=memory,
    )

chain = init_chain()

if "history" not in st.session_state:
    st.session_state.history = []

question = st.chat_input("What is in your mind?")
if question:
    with st.spinner("Thinking..."):
        response = chain(
            {
                "question": question,
                "chat_history": st.session_state.history,   # <- supply it
            }
        )
    st.session_state.history.append((question, response["answer"]))


for user, bot in reversed(st.session_state.history):
    st.markdown(f"**You:** {user}")
    st.markdown(bot)
PY
echo "app.py written"


app.py written


Run `streamlit run app.py` from your terminal and go to http://localhost:8501 to see the chatbot.